# 듣다 CLAP 임베딩 파이프라인

**한 번에 실행**: 좌측 🔑 Secrets 에 두 개 등록 후 (메뉴) → **런타임** → **모두 실행**

## Secrets (필수, 좌측 🔑 아이콘)
1. `SUPABASE_URL` = `https://nsoesrvwkxqifjcxzvol.supabase.co`
2. `SUPABASE_SERVICE_ROLE_KEY` = (Supabase Dashboard → Settings → API → service_role 의 secret)

두 비밀 모두 **'노트북 액세스' 토글 ON** 필수.

## 런타임
메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 (이미 default 면 OK).

## 예상 시간
- 설치 + 모델 로드: ~5분
- 트랙 임베딩 (860개): ~20분
- 매장 archetype (27개): ~30초

## 검증
끝나면 https://www.deudda.com → admin → AI 큐레이션 → 임베딩(PoC) 탭에서 `track_embeddings: 860`, `store_archetypes: 27`, `차원: 512` 확인.

In [ ]:
# 셀 1: 의존성 설치 (2~3분)
!pip install -q transformers librosa soundfile requests

In [ ]:
# 셀 2: 스크립트 다운로드 + Supabase 인증
import urllib.request, os
from google.colab import userdata

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/freemilesarea-boop/SRR-Playlist/claude/playlist-mvp-development-2JmTJ/notebooks/clap_embedding_pipeline.py',
    'clap_embedding_pipeline.py'
)
os.environ['SUPABASE_URL'] = userdata.get('SUPABASE_URL')
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
print('인증/스크립트 준비 OK')

In [ ]:
# 셀 3: CLAP 모델 로드 (~2분, 1.5GB 다운로드)
from clap_embedding_pipeline import load_clap_model
ctx = load_clap_model()

In [ ]:
# 셀 4: 트랙 임베딩 (15~25분, 진행 로그 출력)
from clap_embedding_pipeline import process_track_batch
result = process_track_batch(
    ctx,
    os.environ['SUPABASE_URL'],
    os.environ['SUPABASE_SERVICE_ROLE_KEY'],
    limit=1000
)
print(result)

In [ ]:
# 셀 5: 매장 archetype (text prompt → 27개 임베딩)
from clap_embedding_pipeline import build_store_archetypes_from_text
result = build_store_archetypes_from_text(
    ctx,
    os.environ['SUPABASE_URL'],
    os.environ['SUPABASE_SERVICE_ROLE_KEY']
)
print(result)

## ✅ 완료

https://www.deudda.com → admin → **AI 큐레이션** → **임베딩(PoC)** 탭에서 확인:
- `track_embeddings`: 860 (또는 유사)
- `store_archetypes`: 27
- `차원`: 512

안 보이면 페이지 새로고침.